<a href="https://colab.research.google.com/github/vccf/Deep-Learning-Experiments/blob/drafts-YOLOv5/Demo_YOLOv5_EigenCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install torch torchvision
!pip install opencv-python matplotlib
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 75.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=a9b614b51c10c0c22faaacb431eb8ab2b23e057da3fa3148f0cd8466bdd5d359
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

from ultralytics import YOLO

from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, scale_cam_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO("yolov8n.pt")  # pretrained COCO model

In [ ]:
img_path = "https://ultralytics.com/images/bus.jpg"

import urllib.request
urllib.request.urlretrieve(img_path, "img.jpg")

img = cv2.imread("img.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

rgb_img = img.astype(np.float32) / 255.0

In [ ]:
class YOLOWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.model

    def forward(self, x):
        return self.model(x)

wrapped_model = YOLOWrapper(model)
wrapped_model.eval()

YOLOWrapper(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_ru

In [ ]:
base_model = model.model.model  # real nn.Module list

for i, layer in enumerate(base_model):
    print(i, layer.__class__.__name__)

0 Conv
1 Conv
2 C2f
3 Conv
4 C2f
5 Conv
6 C2f
7 Conv
8 C2f
9 SPPF
10 Upsample
11 Concat
12 C2f
13 Upsample
14 Concat
15 C2f
16 Conv
17 Concat
18 C2f
19 Conv
20 Concat
21 C2f
22 Detect


In [ ]:
target_layers = [base_model[-2]]

In [ ]:
model = YOLO("yolov8n.pt")

wrapped_model = model.model  # DetectionModel
base_model = wrapped_model.model  # nn.Sequential list

target_layers = [base_model[-2]]

In [ ]:
cam = EigenCAM(model=base_model, target_layers=target_layers)

grayscale_cam = cam(input_tensor=torch.tensor(rgb_img).permute(2,0,1).unsqueeze(0))[0]

TypeError: cat() received an invalid combination of arguments - got (Tensor, int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)


In [ ]:
from ultralytics import YOLO
import torch

yolo = YOLO("yolov8n.pt")
model = yolo.model
model.eval()

DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C2f(
      (cv1): Conv(
        (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
    

In [ ]:
target_layers = [model.model[-2]]

In [ ]:
class YOLOCAMWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)[0]  # ONLY raw features/logits

In [ ]:
wrapped_model = YOLOCAMWrapper(model)
wrapped_model.eval()

YOLOCAMWrapper(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track

In [ ]:
from pytorch_grad_cam import EigenCAM

cam = EigenCAM(
    model=wrapped_model,
    target_layers=target_layers
)

input_tensor = torch.tensor(rgb_img).permute(2,0,1).unsqueeze(0).float()

grayscale_cam = cam(input_tensor=input_tensor)[0]

RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 52 but got size 51 for tensor number 1 in the list.

In [ ]:
cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

plt.figure(figsize=(10,10))
plt.imshow(cam_image)
plt.axis("off")
plt.show()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

metrics = model.val(data="coco128.yaml")  # or your dataset yaml

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

WARNING ⚠️ Dataset 'coco128.yaml' images not found, missing path '/content/datasets/coco128/images/train2017'
Unzipping /content/datasets/coco128.zip to /content/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 639.0files/s 0.4s
Dataset download success ✅ (1.1s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1192.4±412.8 MB/s, size: 45.1 KB)
val: Scanning /content/datasets/coco128/labels/train2017... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 288.9it/s 0.4s
val: New cache created: /content/datasets/coco128/labels/train2017.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.5s/it 36.1s
                   all        128        929      0.639      0.536      0.605      0.445
                person         6

In [ ]:
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)

Precision: 0.638501376988649
Recall: 0.5360859555477473
mAP@0.5: 0.6053902973554924
mAP@0.5:0.95: 0.4454218241448946


In [ ]:
import matplotlib.pyplot as plt

metrics.confusion_matrix.plot()
plt.show()

In [ ]:
metrics.save_dir

PosixPath('/content/runs/detect/val')

In [ ]:
print('/content/runs/detect/val/confusion_matrix.png')

/content/runs/detect/val/confusion_matrix.png


In [ ]:
model.val(data="coco128.yaml", plots=True)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1182.4±329.5 MB/s, size: 40.0 KB)
val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 29.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.2s/it 33.3s
                   all        128        929      0.639      0.536      0.605      0.445
                person         61        254      0.793      0.677      0.761      0.532
               bicycle          3          6      0.514      0.333      0.314      0.261
                   car         12         46      0.813      0.217      0.265      0.162
            motorcycle          4          5      0.687      0.887      0.898      0.717
              airplane          5          6       0.82      0.833      0.927      0.672
                   bus     

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79424a5a9730>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
model.val(data="coco128.yaml", plots=True)

Ultralytics 8.4.83 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 807.1±579.2 MB/s, size: 61.6 KB)
val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 21.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.3s/it 34.2s
                   all        128        929      0.639      0.536      0.605      0.445
                person         61        254      0.793      0.677      0.761      0.532
               bicycle          3          6      0.514      0.333      0.314      0.261
                   car         12         46      0.813      0.217      0.265      0.162
            motorcycle          4          5      0.687      0.887      0.898      0.717
              airplane          5          6       0.82      0.833      0.927      0.672
                   bus      

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79424a5aae40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
from ultralytics.utils.ops import letterbox
import numpy as np
import torch

img_resized = letterbox(rgb_img, new_shape=640, stride=32, auto=False)[0]

img_resized = img_resized.transpose(2, 0, 1)  # HWC → CHW
img_resized = np.ascontiguousarray(img_resized)

input_tensor = torch.from_numpy(img_resized).float().unsqueeze(0) / 255.0

model = model.to(input_tensor.device)
model.eval()

class YOLOFeatureWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.model

    def forward(self, x):
        # return ONLY features before detection head
        return self.model(x)

wrapped_model = YOLOFeatureWrapper(model)

target_layers = [wrapped_model.model[-2]]

from pytorch_grad_cam import EigenCAM

cam = EigenCAM(
    model=wrapped_model,
    target_layers=target_layers
)

grayscale_cam = cam(input_tensor=input_tensor)[0]
🧠 WHY THIS ERROR KEPT HAPPENING

You were mixing:

Component	Problem
raw RGB tensor	❌ no letterbox
YOLOv8 model	❌ expects structured input
EigenCAM hook	❌ bypasses preprocessing
ultralytics internals	❌ shape-sensitive concat ops

In [ ]:
from ultralytics.utils.ops import letterbox
import numpy as np
import torch

img_resized = letterbox(rgb_img, new_shape=640, stride=32, auto=False)[0]

img_resized = img_resized.transpose(2, 0, 1)  # HWC → CHW
img_resized = np.ascontiguousarray(img_resized)

input_tensor = torch.from_numpy(img_resized).float().unsqueeze(0) / 255.0

ImportError: cannot import name 'letterbox' from 'ultralytics.utils.ops' (/usr/local/lib/python3.12/dist-packages/ultralytics/utils/ops.py)